### **SilverWork_incremental_industrial_v2**
Incremental Silver processing using only new bronze rows.

**### Step 1- Imports and Setup**

This cell imports spark, window and delta helpers, switches to the right catlog, makes sure the silver schema exists and creates **silver_run_id** for the current run.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid


In [0]:
spark.sql ("use catalog databricks_dev ")
spark.sql("create schema if not exists `01_silver`")
silver_run_id = str(uuid.uuid4())
print("current_silver_run_id:",silver_run_id)


### **Step 2 - Silver control table**
This table stores the latest processing state for each each entity.

It helps us to track

- the latest bronze run already processed by silver
- the latest bronze ingestion timestamp already processed
- how many rows were merged in the latest silver run

In [0]:
spark.sql("""
          CREATE TABLE IF NOT EXISTS `01_silver`.`processing_control` (
            layer string,
            entity_name string,
            last_processed_bronze_run_id string,
            last_processed_ingested_at TIMESTAMP,
            rows_merged bigint,
            run_status string,
            silver_run_id string,
            updated_at TIMESTAMP
          )
          using delta
          """)


### **Step 3 - Helper functions**

This cell contains reusable logic for silver:

- **upsert_to_silver**() merges cleaned/transformed rows into the Silver target table
- **get_latest_processed_bronze_ingested_at()** reads silver watermark
- **upsert_silver_control**() updates silver control table
- **get_incremental_bronze()** reads only new bronze rows that Silver has not yet processed

In [0]:
def upsert_to_silver(df_source, target_table, join_key):
    # Upsert df_source into target_table using join_key; create table if it doesn't exist
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        dt.alias("target").merge(
            df_source.alias("source"),
            f"target.{join_key} = source.{join_key}"
        ).whenMatchedUpdateAll(
        ).whenNotMatchedInsertAll(
        ).execute()
    else:
        df_source.write.format("delta").saveAsTable(target_table)

In [0]:
def get_latest_processed_bronze_ingested_at(entity_name: str):
    # Query the processing_control table for the latest successful run for the given entity in the silver layer
    ctrl = (
        spark.table("databricks_dev.`01_silver`.processing_control")
        .filter(
            (F.col("layer") == "silver") &
            (F.col("entity_name") == entity_name) &
            (F.col("run_status") == "Success")
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )
    rows = ctrl.collect()
    if not rows:
        # If no previous run found, return None for both watermark and run_id
        return None, None
    else:
        # Return the last processed bronze ingestion timestamp and run_id
        return rows[0]["last_processed_ingested_at"], rows[0]["last_processed_bronze_run_id"]

In [0]:
def upsert_silver_control(entity_name:str, last_processed_bronze_run_id:str, last_processed_bronze_ingested_at:datetime, rows_merged:int):
    # Create a DataFrame with the latest processing state for the control table
    ctrl_df = spark.createDataFrame(
        [(
            "silver",
            entity_name,
            last_processed_bronze_run_id,
            last_processed_bronze_ingested_at,
            int(rows_merged),
            "Success",
            silver_run_id,
            datetime.utcnow()
        )],
        schema="""
        layer string,
        entity_name string,
        last_processed_bronze_run_id string,
        last_processed_ingested_at TIMESTAMP,
        rows_merged bigint,
        run_status string,
        silver_run_id string,
        updated_at TIMESTAMP
        """
        
    )

    # Upsert the control DataFrame into the processing_control Delta table
    dt = DeltaTable.forName(spark, "databricks_dev.`01_silver`.processing_control")
    dt.alias("target").merge(
        ctrl_df.alias("source"),
        "target.layer=source.layer and target.entity_name = source.entity_name"
    ).whenMatchedUpdateAll(
    ).whenNotMatchedInsertAll(
    ).execute()

In [0]:
def get_incremental_bronze(bronze_table,entity_name):
    # Get the latest bronze ingestion timestamp and run id already processed by Silver for this entity
    last_processed_bronze_ingested_at,last_run_id = get_latest_processed_bronze_ingested_at(entity_name)
    # Read the full bronze table as a DataFrame
    bronze_df = spark.read.table(bronze_table)
    # If no previous watermark, return all rows
    if last_processed_bronze_ingested_at is None:
        return bronze_df,last_processed_bronze_ingested_at,last_run_id
    
    # Otherwise, filter for only new rows with bronze_ingested_ts greater than the last processed watermark
    return bronze_df.filter(
        F.col("bronze_ingested_ts") > F.lit(last_processed_bronze_ingested_at)
    ), last_processed_bronze_ingested_at,last_run_id

### **Step 4 - Orders incremental processing**

This cell processes Orders from Bronze to silver.

It does following:

- reads only new Bronze order rows
- clean values like **order_status** and **order_amount**
- keeps only latest version per **order_id**
- validates busniness rules
- sends bad rows to quarantine
- merges good rows into **orders_transformed**
- 

In [0]:
df_raw= spark.sql("select * from databricks_dev.00_bronze.orders_raw")
display(df_raw)

In [0]:
# Step 4 - Orders incremental processing

# Read only the new Bronze order rows that Silver has not processed yet.
orders_inc, last_orders_ingested_at,last_order_run_id = get_incremental_bronze("databricks_dev.00_bronze.orders_raw", "orders")

# Count the number of incremental order rows entering Silver in this run.
orders_inc_count = orders_inc.count()
print(f"orders rows entering silver in this run: {orders_inc_count}")

# Only run Silver order cleaning and validation if there are new Bronze rows.
if orders_inc_count > 0:
    # Define a window to keep only the latest version per order_id based on updated_at and bronze_ingested_ts
    orders_window = Window.partitionBy("order_id").orderBy(
        F.col("updated_at").cast("timestamp").desc(),
        F.col("bronze_ingested_ts").desc()
    )

    # Clean and standardize order_status and order_amount, convert timestamps, and keep latest version per order_id
    orders_cleaned = (
        orders_inc
        .withColumn("order_status", F.upper(F.trim(F.col("order_status"))))  # Standardize order_status to uppercase and trim spaces
        .withColumn("order_status", F.when(F.col("order_status") == '', F.lit("None")).otherwise(F.col("order_status")))  # Replace empty order_status with 'None'
        .withColumn("order_amount", F.regexp_replace(F.col("order_amount").cast("string"), r"[^0-9\.]", ""))  # Remove non-numeric characters from order_amount
        .withColumn("order_amount", F.regexp_replace(F.col("order_amount"), r",", "."))  # Replace comma with dot in order_amount
        .withColumn("order_amount", F.trim(F.col("order_amount")))  # Trim spaces in order_amount
        .withColumn("order_amount", F.when(F.col("order_amount").isin("", "N/A", "NULL", "??"), F.lit(None)).otherwise(F.col("order_amount")))  # Replace invalid values with null
        .withColumn("order_amount", F.when(F.col("order_amount") == "", F.lit(None)).otherwise(F.col("order_amount")))  # Replace empty string with null before cast
        .withColumn("order_amount", F.col("order_amount").cast("double"))  # Cast order_amount to double
        .withColumn("created_at", F.to_timestamp("created_at"))  # Convert created_at to timestamp
        .withColumn("updated_at", F.to_timestamp("updated_at"))  # Convert updated_at to timestamp
        .withColumn("row_number", F.row_number().over(orders_window))  # Assign row number within window
        .filter(F.col("row_number") == 1)  # Keep only the latest version per order_id
        .drop("row_number")
        .withColumn("silver_run_id", F.lit(silver_run_id))  # Add current silver_run_id for traceability
    )

    # Split orders: orders with order_amount <= 0 go to quarantine, others continue
    orders_to_quarantine = orders_cleaned.filter(F.col("order_amount").isNull() | (F.col("order_amount") <= 0))
    orders_to_process = orders_cleaned.filter(F.col("order_amount").isNotNull() & (F.col("order_amount") > 0))

    # Merge cleaned, valid orders into the Silver cleaned table
    upsert_to_silver(orders_to_process, "databricks_dev.01_silver.orders_cleaned", "order_id")

    # Validate business rules and add derived columns for reporting and DQ checks
    orders_validated = (
        orders_to_process
        .withColumn(
            "to_be_verified_by_orders_team_dq_issue",
            F.when(F.col("customer_id").isNull(), "verify_customer_id")
            .when(F.col("product_id").isNull(), "verify_product_id")
            .when(F.col("order_status").isNull() | (F.trim(F.col("order_status")) == ""), "verify_order_status")
            .otherwise("No issues")
        )  # Flag rows with DQ issues
        .withColumn("check_order_amount", F.lit("False"))  # All here are valid
        .withColumn("order_date", F.to_date("created_at"))  # Extract order date
        .withColumn("order_year", F.year("created_at"))  # Extract order year
        .withColumn("order_month", F.month("created_at"))  # Extract order month
        .withColumn("order_day", F.dayofmonth("created_at"))  # Extract order day
        .withColumn("order_dow", F.date_format("created_at", "E"))  # Extract day of week
    )

    # Filter good orders (no DQ issues)
    orders_good = orders_validated.filter(F.col("to_be_verified_by_orders_team_dq_issue") == "No issues")

    # Filter and prepare quarantined orders (with DQ issues)
    orders_quarantined = (
        orders_validated
        .filter(F.col("to_be_verified_by_orders_team_dq_issue") != "No issues")
        .withColumn("quarantine_ts", F.current_timestamp())  # Add quarantine timestamp
        .withColumn("quarantine_run_id", F.lit(silver_run_id))  # Add quarantine run id
        .withColumn("quarantine_status", F.lit("quarantined"))  # Set quarantine status
    )

    # Prepare orders with order_amount <= 0 for quarantine
    orders_to_quarantine = (
        orders_to_quarantine
        .withColumn("to_be_verified_by_orders_team_dq_issue", F.lit("verify_order_amount"))
        .withColumn("check_order_amount", F.lit("True"))
        .withColumn("order_date", F.to_date("created_at"))
        .withColumn("order_year", F.year("created_at"))
        .withColumn("order_month", F.month("created_at"))
        .withColumn("order_day", F.dayofmonth("created_at"))
        .withColumn("order_dow", F.date_format("created_at", "E"))
        .withColumn("quarantine_ts", F.current_timestamp())
        .withColumn("quarantine_run_id", F.lit(silver_run_id))
        .withColumn("quarantine_status", F.lit("quarantined"))
    )

    # Merge good orders into the Silver transformed table
    upsert_to_silver(orders_good, "databricks_dev.01_silver.orders_transformed", "order_id")

    # Append all bad order rows to quarantine table (DQ issues + order_amount <= 0)
    orders_quarantine_final = orders_quarantined.unionByName(orders_to_quarantine)
    orders_quarantine_final.write.format("delta").mode("append").saveAsTable("databricks_dev.01_silver.orders_quarantined")

    # Get the max bronze_ingested_at and corresponding bronze_run_id for control table update
    mx_ingested = orders_inc.agg(F.max("bronze_ingested_ts").alias("mx_ingested")).collect()[0]["mx_ingested"]
    mx_run = (
        orders_inc
        .filter(F.col("bronze_ingested_ts") == F.lit(mx_ingested))
        .agg(F.max("bronze_run_id").alias("mx_run"))
        .collect()[0]["mx_run"]
    )

    # Update the Silver control table with the latest processing state
    upsert_silver_control("orders", mx_run, mx_ingested, orders_good.count())

else:
    # If no new rows, log and update control table accordingly
    print("No new orders bronze rows to process in this run.")
    upsert_silver_control("orders", last_order_run_id, last_orders_ingested_at, orders_inc_count)

In [0]:
%sql
select * from databricks_dev.`01_silver`.orders_transformed

In [0]:
# Step 5 - Product incremental Processing

# Read only new Bronze product rows not yet processed by Silver
product_inc, last_product_ingested_at,last_product_run_id = get_incremental_bronze("databricks_dev.00_bronze.products_raw", "product")
# Count the number of incremental product rows entering Silver in this run
product_inc_count = product_inc.count()

if product_inc_count > 0:
    # Define a window to keep only the latest version per product_id based on updated_at and bronze_ingested_ts
    product_window = Window.partitionBy("product_id").orderBy(
        F.col("updated_at").cast("timestamp").desc(),
        F.col("bronze_ingested_ts").desc()
    )

    # Clean and standardize product_name, category, price, convert timestamps, and keep latest version per product_id
    product_cleaned = (
        product_inc
        .withColumn("product_name", F.upper(F.trim(F.col("product_name"))))  # Standardize product_name to uppercase and trim spaces
        .withColumn("product_name", F.regexp_replace(F.col("product_name"), r"-", " "))  # Replace hyphens with spaces in product_name
        .withColumn("product_name", F.trim(F.col("product_name")))  # Trim again after hyphen replacement
        .withColumn(
            "product_name",
            F.when(
                F.regexp_replace(F.col("product_name"), r"[^a-zA-Z0-9 ]", "") == "",
                F.lit(None)
            ).otherwise(F.col("product_name"))
        )  # Set product_name to None if only invalid chars
        .withColumn("category", F.upper(F.trim(F.col("category"))))  # Standardize category to uppercase and trim spaces
        .withColumn(
            "category",
            F.when(
                F.upper(F.trim(F.col("category"))).contains("ELECTRNICS"),
                F.lit("ELECTRONICS")
            ).otherwise(F.col("category"))
        )  # Correct misspelled ELECTRNICS to ELECTRONICS
        .withColumn("price", F.regexp_replace(F.col("price").cast("string"), r"[^0-9\.]", ""))  # Remove non-numeric chars from price
        .withColumn("price", F.regexp_replace(F.col("price"), r",", "."))  # Replace comma with dot in price
        .withColumn("price", F.regexp_replace(F.col("price"), r"\s+", ""))  # Remove whitespace from price
        .withColumn("price", F.when(F.col("price") == "", F.lit(None)).otherwise(F.col("price")))  # Replace empty string with null before cast
        .withColumn("price", F.col("price").cast("double"))  # Cast price to double
        .withColumn("updated_at", F.to_timestamp("updated_at"))  # Convert updated_at to timestamp
        .withColumn("row_number", F.row_number().over(product_window))  # Assign row number within window
        .filter(F.col("row_number") == 1)  # Keep only the latest version per product_id
        .drop("row_number")  # Drop row_number column
        .withColumn("silver_run_id", F.lit(silver_run_id))  # Add current silver_run_id for traceability
    )

    # Merge cleaned products into the Silver cleaned table
    upsert_to_silver(product_cleaned, "databricks_dev.01_silver.product_cleaned", "product_id")

    # Validate business rules and add DQ issue flags
    Product_validated = (
        product_cleaned
        .withColumn(
            "to_be_verified_by_product_team_dq_issue",
            F.when(F.col("product_name").isNull(), ("verify product_name_null"))
            .when(F.col("category").isNull(), ("verify category_null"))
            .when(F.col("price").isNull() | (F.col("price") <= 0), ("verify price <=0"))
            .otherwise("No issues"))
        .withColumn(
            "check_product_price", 
            F.when(F.col("price").isNull() | (F.col("price") <= 0), ("invalid price"))
            .otherwise("Valid price")
        )
    )

    # Keep only valid product rows for transformed silver table
    product_good = (
        Product_validated
        .filter(
            (F.col("to_be_verified_by_product_team_dq_issue") == "No issues") &
            (F.col("check_product_price") == "Valid price")
        )
    )
    # Drop price_raw column if present
    if "price_raw" in product_good.columns:
        product_good = product_good.drop("price_raw")

    # Merge valid products into the Silver transformed table
    upsert_to_silver(product_good, "databricks_dev.01_silver.product_transformed", "product_id")

    # Keep only invalid product rows for quarantine table
    product_to_quarantine = (
        Product_validated
        .filter(
            (F.col("to_be_verified_by_product_team_dq_issue") != "No issues") |
            (F.col("check_product_price") != "Valid price")
        )
        .withColumn("quarantine_ts", F.current_timestamp())  # Add quarantine timestamp
    )

    # Merge invalid rows to quarantine delta table
    upsert_to_silver(product_to_quarantine, "databricks_dev.01_silver.product_quarantined", "product_id")

    # Append bad data to quarantine table
    product_to_quarantine.write.format("delta").mode("append").saveAsTable("databricks_dev.01_silver.product_quarantined")

    # Get max bronze_ingested_ts and corresponding bronze_run_id for control table update
    mx_ingested = product_inc.agg(F.max("bronze_ingested_ts").alias("mx_ingested")).collect()[0]["mx_ingested"]
    mx_run = (
        product_inc
        .filter(F.col("bronze_ingested_ts") == F.lit(mx_ingested))
        .agg(F.max("bronze_run_id").alias("mx_run"))
        .collect()[0]["mx_run"]
    )
    # Update Silver control table with latest processing state
    upsert_silver_control("product", mx_run, mx_ingested, product_good.count())

else:
    # If no new rows, log and update control table accordingly
    print("no new product bronze rows for silver to process in this run")
    upsert_silver_control("product", last_product_run_id, last_product_ingested_at, product_inc_count)

In [0]:
%sql 
select * from databricks_dev.`01_silver`.product_cleaned

In [0]:
%sql
select * from databricks_dev.`01_silver`.product_transformed

### **Step 6 - Payment incremental processing**
This cell processes **payment** from bronze to silver

It cleans:

-  payment_status
-  paid_amount
-  processed_at

Then it validates records, quarantines bad rows and merges valid rows into the silver transformed payments table.

In [0]:
# Step 6 - Payments incremental processing

# Read only new Bronze payment rows not yet processed by Silver
payment_inc, last_payment_ingested_at,last_payment_run_id = get_incremental_bronze("databricks_dev.00_bronze.payments_raw", "payments")
payment_inc_count = payment_inc.count()
print(f"payment rows entering silver in this run: {payment_inc_count}")

if payment_inc_count > 0:
    # Define a window to keep only the latest version per payment_id based on processed_at and bronze_ingested_ts
    payment_window = Window.partitionBy("payment_id").orderBy(
        F.col("processed_at").cast("timestamp").desc(),
        F.col("bronze_ingested_ts").desc()
    )

    # Clean and standardize payment fields
    payment_cleaned = (
        payment_inc
        .withColumn("payment_status", F.upper(F.trim(F.col("payment_status"))))  # Standardize payment_status to uppercase and trim spaces
        .withColumn("payment_status", F.when(F.col("payment_status") == "", F.lit("None")).otherwise(F.col("payment_status")))  # Replace empty payment_status with 'None'
        .withColumn("paid_amount", F.regexp_replace(F.col("paid_amount").cast("string"), r"[^0-9\.]", ""))  # Remove non-numeric chars from paid_amount
        .withColumn("paid_amount", F.regexp_replace(F.col("paid_amount"), r",", "."))  # Replace comma with dot in paid_amount
        .withColumn("paid_amount", F.trim(F.col("paid_amount")))  # Trim spaces in paid_amount
        .withColumn("paid_amount", F.when(F.col("paid_amount").isin("", "N/A", "NULL", "??"), F.lit(None)).otherwise(F.col("paid_amount")))  # Replace invalid values with null
        .withColumn("paid_amount", F.when(F.col("paid_amount") == "", F.lit(None)).otherwise(F.col("paid_amount")))  # Replace empty string with null before cast
        .withColumn("paid_amount", F.col("paid_amount").cast("double"))  # Cast paid_amount to double
        .withColumn("processed_at", F.to_timestamp("processed_at"))  # Convert processed_at to timestamp
        .withColumn("row_number", F.row_number().over(payment_window))  # Assign row number within window
        .filter(F.col("row_number") == 1)  # Keep only the latest version per payment_id
        .drop("row_number")
        .withColumn("silver_run_id", F.lit(silver_run_id))  # Add current silver_run_id for traceability
    )

    # Split payments: paid_amount <= 0 go to quarantine, others continue
    payments_to_quarantine = payment_cleaned.filter(F.col("paid_amount").isNull() | (F.col("paid_amount") <= 0))
    payments_to_process = payment_cleaned.filter(F.col("paid_amount").isNotNull() & (F.col("paid_amount") > 0))

    # Merge cleaned valid payments into the Silver cleaned table
    upsert_to_silver(payments_to_process, "databricks_dev.01_silver.payments_cleaned", "payment_id")

    # Validate business rules and add DQ issue flags
    payments_validated = (
        payments_to_process
        .withColumn(
            "to_be_verified_by_payments_team_dq_issue",
            F.when(F.col("order_id").isNull(), "verify_order_id")
            .when(F.col("payment_status").isNull() | (F.trim(F.col("payment_status")) == ""), "verify_payment_status")
            .otherwise("No issues")
        )  # Flag rows with DQ issues
        .withColumn("check_paid_amount", F.lit("False"))  # All here are valid
        .withColumn("payment_date", F.to_date("processed_at"))  # Extract payment date
        .withColumn("payment_year", F.year("processed_at"))  # Extract payment year
        .withColumn("payment_month", F.month("processed_at"))  # Extract payment month
        .withColumn("payment_day", F.dayofmonth("processed_at"))  # Extract payment day
        .withColumn("payment_dow", F.date_format("processed_at", "E"))  # Extract day of week
    )

    # Filter good payments (no DQ issues)
    payments_good = payments_validated.filter(F.col("to_be_verified_by_payments_team_dq_issue") == "No issues")

    # Filter and prepare quarantined payments (with DQ issues)
    payments_quarantined = (
        payments_validated
        .filter(F.col("to_be_verified_by_payments_team_dq_issue") != "No issues")
        .withColumn("quarantine_ts", F.current_timestamp())  # Add quarantine timestamp
        .withColumn("quarantine_run_id", F.lit(silver_run_id))  # Add quarantine run id
        .withColumn("quarantine_status", F.lit("quarantined"))  # Set quarantine status
    )

    # Prepare paid_amount <= 0 rows for quarantine
    payments_to_quarantine = (
        payments_to_quarantine
        .withColumn("to_be_verified_by_payments_team_dq_issue", F.lit("verify_paid_amount"))
        .withColumn("check_paid_amount", F.lit("True"))
        .withColumn("payment_date", F.to_date("processed_at"))
        .withColumn("payment_year", F.year("processed_at"))
        .withColumn("payment_month", F.month("processed_at"))
        .withColumn("payment_day", F.dayofmonth("processed_at"))
        .withColumn("payment_dow", F.date_format("processed_at", "E"))
        .withColumn("quarantine_ts", F.current_timestamp())
        .withColumn("quarantine_run_id", F.lit(silver_run_id))
        .withColumn("quarantine_status", F.lit("quarantined"))
    )

    # Merge good payments into the Silver transformed table
    upsert_to_silver(payments_good, "databricks_dev.01_silver.payments_transformed", "payment_id")

    # Append all bad payment rows to quarantine table (DQ issues + paid_amount <= 0)
    payments_quarantine_final = payments_quarantined.unionByName(payments_to_quarantine)
    payments_quarantine_final.write.format("delta").mode("append").saveAsTable("databricks_dev.01_silver.payments_quarantined")

    # Get the max bronze_ingested_ts and corresponding bronze_run_id for control table update
    mx_ingested = payment_inc.agg(F.max("bronze_ingested_ts").alias("mx_ingested")).collect()[0]["mx_ingested"]
    mx_run = (
        payment_inc
        .filter(F.col("bronze_ingested_ts") == F.lit(mx_ingested))
        .agg(F.max("bronze_run_id").alias("mx_run"))
        .collect()[0]["mx_run"]
    )

    # Update the Silver control table with the latest processing state
    upsert_silver_control("payments", mx_run, mx_ingested, payments_good.count())

else:
    # If no new rows, log and update control table accordingly
    print("no new payment bronze rows for silver to process in this run")
    upsert_silver_control("payments", last_payment_run_id, last_payment_ingested_at, payment_inc_count)

In [0]:
print("product transformed count: ", spark.sql("select count(*) from databricks_dev.01_silver.product_transformed").collect()[0][0])
print("order transformed count: ", spark.sql("select count(*) from databricks_dev.01_silver.orders_transformed").collect()[0][0])
print("payment transformed count: ", spark.sql("select count(*) from databricks_dev.01_silver.payments_transformed").collect()[0][0])
display(spark.table("databricks_dev.`01_silver`.processing_control").orderBy("entity_name"))
